# RiseOfTFMs: working with tabular foundation models

A hands-on tour of how the pre-modeling pipeline changes when you swap a gradient-boosting
model for **TabPFN**. For practitioners who already have solid instincts for scaling, missing
values, categoricals, and feature engineering, and want to see what still applies and what
no longer does.

The cut: **ground in the familiar practice first, then show what TabPFN changes, then go under
the hood.** This notebook is the runnable backing for the post series.

> Run on the `.venv` kernel where `tabpfn==8.0.3` lives.

## Post 0: the baseline ritual, retired

**The familiar checklist.** Starting a tabular problem, the reflex is a fixed ritual: split the
data, then give each model the preprocessing it demands (impute missing values, encode
categoricals, scale for linear and distance models), reach for a gradient-boosting model, and
tune it. How much preprocessing you owe depends on the model: logistic regression needs all of
it, LightGBM and XGBoost handle missing values and (with care) categoricals, CatBoost takes
categoricals natively.

**What TabPFN does.** Fit and predict on the raw frame: no imputation, no encoding, no scaling,
no tuning. Below we put it next to the incumbents on the same data and the same row budget,
first on one dataset, then across a binary panel.

In [1]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score

from tabulate import tabulate
def print_df(df, tablefmt="psql", showindex=False):
    print(tabulate(df, headers="keys", tablefmt=tablefmt, showindex=showindex))

import torch
from tqdm import tqdm
import tabpfn
from tabpfn import TabPFNClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

print("tabpfn", tabpfn.__version__, "| cuda", torch.cuda.is_available())

/home/jovyan/doValueClassifier/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


tabpfn 8.0.3 | cuda True


### Toolkit

A few small, reusable helpers: load a binary OpenML dataset (categoricals stay `category`
dtype, missing stays NaN), make a capped stratified split, an Expected Calibration Error, and a
uniform `evaluate` that times fit and predict and returns AUC / AUPRC / ECE.

In [2]:
def load_openml(data_id, positive_label=None):
    """Binary OpenML dataset -> (X df, y in {0,1}). Categoricals stay category
    dtype and missing stays NaN, so models that accept raw frames get them untouched."""
    ds = fetch_openml(data_id=data_id, as_frame=True)
    X, y = ds.data.copy(), ds.target
    pos = positive_label if positive_label is not None else sorted(y.unique())[-1]
    return X, (y == pos).astype(int)

def split_and_cap(X, y, train_cap=10000, test_cap=10000, seed=0):
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=0.25, stratify=y, random_state=seed)
    def cap(Xd, yd, n):
        if len(Xd) <= n:
            return Xd, yd
        idx = Xd.sample(n, random_state=seed).index
        return Xd.loc[idx], yd.loc[idx]
    X_tr, y_tr = cap(X_tr, y_tr, train_cap)
    X_te, y_te = cap(X_te, y_te, test_cap)
    return X_tr, X_te, y_tr, y_te

def ece(y_true, p, n_bins=15):
    """Expected Calibration Error (equal-width bins on the positive prob)."""
    idx = np.clip(np.digitize(p, np.linspace(0, 1, n_bins + 1)) - 1, 0, n_bins - 1)
    e = 0.0
    for b in range(n_bins):
        m = idx == b
        if m.any():
            e += m.mean() * abs(y_true[m].mean() - p[m].mean())
    return e

def evaluate(model, X_tr, y_tr, X_te, y_te, label):
    t0 = time.time(); model.fit(X_tr, y_tr); fit_s = time.time() - t0
    t0 = time.time(); p = model.predict_proba(X_te)[:, 1]; pred_s = time.time() - t0
    yv = np.asarray(y_te)
    return {"model": label, "auc": roc_auc_score(yv, p),
            "auprc": average_precision_score(yv, p), "ece": ece(yv, p),
            "fit_s": round(fit_s, 2), "predict_s": round(pred_s, 2)}

### The 30-second baseline (one dataset)

`adult` (OpenML 1590), the target is income > 50K. Both models see the raw frame: TabPFN by
design, LightGBM via its native NaN and category-dtype handling. Same split, same `<=10k`
train rows.

In [3]:
X, y = load_openml(1590)
X_tr, X_te, y_tr, y_te = split_and_cap(X, y)
print(f"train {X_tr.shape}  test {X_te.shape}  positive rate {y_tr.mean():.3f}")

rows = []
rows.append(evaluate(TabPFNClassifier(random_state=0), X_tr, y_tr, X_te, y_te, "tabpfn_raw"))
rows.append(evaluate(LGBMClassifier(random_state=0, verbose=-1), X_tr, y_tr, X_te, y_te, "lgbm_default"))
print_df(pd.DataFrame(rows).set_index("model").round(4), showindex=True)

train (10000, 14)  test (10000, 14)  positive rate 0.245
+--------------+--------+---------+--------+---------+-------------+
| model        |    auc |   auprc |    ece |   fit_s |   predict_s |
|--------------+--------+---------+--------+---------+-------------|
| tabpfn_raw   | 0.9148 |  0.7869 | 0.011  |    1.25 |       14.03 |
| lgbm_default | 0.9228 |  0.8156 | 0.0095 |    0.12 |        0.01 |
+--------------+--------+---------+--------+---------+-------------+


### Across the binary panel

The same roster on six binary OpenML datasets. Every model gets the **same `<=10k` train rows**
(a matched budget), and each non-TabPFN model gets the *minimal* preprocessing it needs: TabPFN
nothing, LightGBM and XGBoost native categoricals, CatBoost with its categorical NaNs filled,
the linear model the full impute + scale + one-hot pipeline. High-cardinality columns are left
at their native dtype here; that question belongs to Post 3.

In [4]:
def make_linear(cat_cols, num_cols):
    transformers = [("num", Pipeline([("imp", SimpleImputer(strategy="median")),
                                      ("sc", StandardScaler())]), num_cols)]
    if cat_cols: 
        transformers.append(("cat", Pipeline([
            ("imp", SimpleImputer(strategy="most_frequent")),
            ("oh", OneHotEncoder(handle_unknown="infrequent_if_exist", max_categories=50))]), cat_cols))
    return Pipeline([("pre", ColumnTransformer(transformers)),
                     ("clf", LogisticRegression(max_iter=2000))])

def make_catboost(cat_cols):
    def fill(df):
        df = df.copy()
        for c in cat_cols:
            df[c] = df[c].astype("object").fillna("missing").astype(str)
        return df
    return Pipeline([("fill", FunctionTransformer(fill)),
                     ("clf", CatBoostClassifier(cat_features=cat_cols, random_state=0, verbose=0))])

def make_xgb():
    return XGBClassifier(enable_categorical=True, tree_method="hist",
                         random_state=0, eval_metric="logloss")

def prep_dataset(data_id, train_cap=10000, test_cap=10000, seed=0):
    X, y = load_openml(data_id)
    X_tr, X_te, y_tr, y_te = split_and_cap(X, y, train_cap, test_cap, seed)
    cat_cols = X_tr.select_dtypes(exclude="number").columns.tolist()
    num_cols = X_tr.select_dtypes("number").columns.tolist()
    for c in cat_cols:  # align test categories to train for native handling
        X_tr[c] = X_tr[c].astype("category")
        X_te[c] = pd.Categorical(X_te[c], categories=X_tr[c].cat.categories)
    return X_tr, X_te, y_tr, y_te, cat_cols, num_cols

In [5]:
PANEL = {"credit-g": 31, "adult": 1590, "bank-marketing": 1461,
         "electricity": 151, "click-prediction": 1220, "bioresponse": 4134}

def build_roster(cat_cols, num_cols):
    return {"tabpfn":   TabPFNClassifier(random_state=0),   # raw, nothing
            "logreg":   make_linear(cat_cols, num_cols),
            "lgbm":     LGBMClassifier(random_state=0, verbose=-1),
            "xgb":      make_xgb(),
            "catboost": make_catboost(cat_cols)}

records = []
for name, data_id in tqdm(PANEL.items()):
    Xtr, Xte, ytr, yte, cat_cols, num_cols = prep_dataset(data_id)
    for mname, model in build_roster(cat_cols, num_cols).items():
        try:
            r = evaluate(model, Xtr, ytr, Xte, yte, mname); r["dataset"] = name
            records.append(r); print(f"{name:16s} {mname:9s} auc={r['auc']:.4f}")
        except Exception as e:
            print(f"{name:16s} {mname:9s} FAILED: {type(e).__name__}: {str(e)[:90]}")
panel = pd.DataFrame(records)

  0%|          | 0/6 [00:00<?, ?it/s]

credit-g         tabpfn    auc=0.8234
credit-g         logreg    auc=0.8162
credit-g         lgbm      auc=0.7944
credit-g         xgb       auc=0.7823


 17%|█▋        | 1/6 [00:07<00:37,  7.51s/it]

credit-g         catboost  auc=0.8188
adult            tabpfn    auc=0.9148
adult            logreg    auc=0.9031
adult            lgbm      auc=0.9228
adult            xgb       auc=0.9161


 33%|███▎      | 2/6 [00:27<00:59, 14.92s/it]

...

click-prediction catboost  auc=0.6799


/home/jovyan/doValueClassifier/.venv/lib/python3.11/site-packages/tabpfn/classifier.py:720: UserWarning: Auto-scaling n_estimators from 8 to 9 so every feature is included in at least one ensemble member (n_total_features=1776, max_features_per_estimator=200). Pass n_estimators >= 9 to silence this warning.
  self.n_estimators_ = scale_n_estimators_for_feature_coverage(


bioresponse      tabpfn    auc=0.8864
bioresponse      logreg    auc=0.7755
bioresponse      lgbm      auc=0.8884
bioresponse      xgb       auc=0.8844


100%|██████████| 6/6 [01:58<00:00, 19.69s/it]

bioresponse      catboost  auc=0.8842


In [6]:
auc_table = (panel.pivot(index="dataset", columns="model", values="auc")
                  .reindex(index=list(PANEL), columns=["logreg", "lgbm", "xgb", "catboost", "tabpfn"])
                  .round(4))
auc_table["best"] = auc_table.idxmax(axis=1)
print_df(auc_table, showindex=True)

+------------------+----------+--------+--------+------------+----------+----------+
| dataset          |   logreg |   lgbm |    xgb |   catboost |   tabpfn | best     |
|------------------+----------+--------+--------+------------+----------+----------|
| credit-g         |   0.8162 | 0.7944 | 0.7823 |     0.8188 |   0.8234 | tabpfn   |
| adult            |   0.9031 | 0.9228 | 0.9161 |     0.925  |   0.9148 | catboost |
| bank-marketing   |   0.9046 | 0.9254 | 0.921  |     0.9295 |   0.9399 | tabpfn   |
| electricity      |   0.8313 | 0.9462 | 0.9503 |     0.9356 |   0.9621 | tabpfn   |
| click-prediction |   0.6276 | 0.6745 | 0.6507 |     0.6799 |   0.6896 | tabpfn   |
| bioresponse      |   0.7755 | 0.8884 | 0.8844 |     0.8842 |   0.8864 | lgbm     |
+------------------+----------+--------+--------+------------+----------+----------+


### What the panel shows

With zero preprocessing and no tuning, TabPFN is the best model on **4 of the 6** datasets
(credit-g, bank-marketing, electricity, click-prediction). On a 5th, bioresponse, it sits
0.002 AUC behind the leader, a statistical tie. It trails clearly on only one, `adult`, where
CatBoost leads by about 0.010.

The asymmetry of effort is the point. The boosters are at their defaults but still got the
preprocessing they need (native categoricals, filled NaNs), and the linear model got the full
impute + scale + one-hot pipeline. TabPFN got the raw frame. The largest margins
(electricity +0.012, bank-marketing +0.010, click-prediction +0.010) are well outside seed
noise.

Two honest caveats: `credit-g` has only ~250 test rows, so its win is within noise, and the
boosters are untuned (tuning tends to move AUC by a few thousandths at this scale). And notice
the linear model: even fully preprocessed it trails the field badly on electricity (0.83 vs
0.96). Scaling and encoding buy a linear model a seat at the table, not a competitive one. We
pick up that thread in Post 1.

In [7]:
# AUC was the headline; the run also measured ranking under imbalance (AUPRC, higher better)
# and calibration (ECE, lower better). TabPFN's out-of-the-box calibration shows up here.
for metric in ["auprc", "ece"]:
  print(f"\n{metric.upper()}  ({'higher' if metric=='auprc' else 'lower'} is better)")
  t = (panel.pivot(index="dataset", columns="model", values=metric)
            .reindex(index=list(PANEL), columns=["logreg", "lgbm", "xgb", "catboost", "tabpfn"])
            .round(4))
  print_df(t, showindex=True)


AUPRC  (higher is better)
+------------------+----------+--------+--------+------------+----------+
| dataset          |   logreg |   lgbm |    xgb |   catboost |   tabpfn |
|------------------+----------+--------+--------+------------+----------|
| credit-g         |   0.9194 | 0.8992 | 0.8853 |     0.9209 |   0.9204 |
| adult            |   0.7564 | 0.8156 | 0.8001 |     0.8188 |   0.7869 |
| bank-marketing   |   0.5569 | 0.5848 | 0.5715 |     0.6029 |   0.6271 |
| electricity      |   0.7999 | 0.9301 | 0.9348 |     0.9162 |   0.9512 |
| click-prediction |   0.2527 | 0.3035 | 0.2818 |     0.3087 |   0.3147 |
| bioresponse      |   0.7747 | 0.9075 | 0.9069 |     0.9045 |   0.8983 |
+------------------+----------+--------+--------+------------+----------+

ECE  (lower is better)
+------------------+----------+--------+--------+------------+----------+
| dataset          |   logreg |   lgbm |    xgb |   catboost |   tabpfn |
|------------------+----------+--------+--------+------------

### Reading AUPRC and calibration

**Under imbalance, AUPRC tracks AUC.** On the two most skewed targets, bank-marketing (~12%
positive) and click-prediction (~17%), TabPFN has the best AUPRC of any model (0.627 and 0.315),
and it also leads electricity. It trails on adult and bioresponse, the same two where its AUC
was not on top. Where the positives are rare and AUPRC is the number you would actually report,
the raw-frame model is in front.
the raw-frame model is in front.

**Calibration is good but not uniquely TabPFN's.** Once CatBoost and XGBoost are included, no
single model owns ECE: TabPFN is best-calibrated on credit-g and bank-marketing, second on
electricity and bioresponse, middling on adult and click-prediction (where everything is already
within ~0.01). The fair claim is consistency: TabPFN is never the worst-calibrated, and its
largest ECE across all six (0.066) is smaller than the calibration blowups the boosters show on
individual datasets (logreg 0.198 on bioresponse, XGBoost 0.147 on credit-g). With no
calibration step, that reliability is the win.

### Under the hood: why fit was instant and predict was not

TabPFN does not train on your data. The network was pretrained once, on millions of synthetic
tables, to do supervised learning in a single forward pass. So `fit` just loads your training
rows as context (about a second), and `predict` runs attention over the entire training set for
every test row. That is why TabPFN's predict times are one to three orders of magnitude larger
than a booster's: it reads the training data at inference, it does not consult fixed weights.

The same design fixes hard limits, the pretraining envelope, on how many samples, features, and
classes one forward pass can take. Both the envelope and the predict-time tax are read off the
fitted model below. The cost of in-context learning is that tax, and it, not accuracy, is
TabPFN's real price. A later post makes it the whole subject.

In [8]:
clf = TabPFNClassifier(random_state=0).fit(X_tr.head(200), y_tr.head(200))
ic = clf.inference_config_
print(f"pretraining envelope -> samples {ic.MAX_NUMBER_OF_SAMPLES:,} | "
    f"features {ic.MAX_NUMBER_OF_FEATURES:,} | classes {ic.MAX_NUMBER_OF_CLASSES}")

print("\npredict time in seconds (TabPFN vs the boosters):")
t = (panel.pivot(index="dataset", columns="model", values="predict_s")
        .reindex(index=list(PANEL), columns=["logreg", "lgbm", "xgb", "catboost", "tabpfn"])
        .round(3))
print_df(t, showindex=True)

pretraining envelope -> samples 1,000,000 | features 2,000 | classes 160

predict time in seconds (TabPFN vs the boosters):
+------------------+----------+--------+-------+------------+----------+
| dataset          |   logreg |   lgbm |   xgb |   catboost |   tabpfn |
|------------------+----------+--------+-------+------------+----------|
| credit-g         |     0.01 |   0    |  0.01 |       0.01 |     0.62 |
| adult            |     0.03 |   0.01 |  0.01 |       0.03 |    11    |
| bank-marketing   |     0.11 |   0.01 |  0.01 |       0.03 |    11.4  |
| electricity      |     0.02 |   0.01 |  0    |       0.01 |    10.17 |
| click-prediction |     0.02 |   0.01 |  0    |       0    |    10.31 |
| bioresponse      |     0.11 |   0.01 |  0.07 |       0.06 |    10.35 |
+------------------+----------+--------+-------+------------+----------+


---

## Post 1: scaling, transforms, and outliers

  **The familiar checklist.** Numeric columns rarely arrive model-ready, so the classical reflex
  is a short sequence of fixes:

  - **Scale** so no feature dominates on units alone: `StandardScaler` (z-score), `MinMaxScaler`
    (to [0, 1]), or `RobustScaler` (median and IQR, for heavy tails).
  - **Transform skew** so a long right tail does not swamp the fit: `log1p` on nonnegative counts
    and money, or a power transform (Box-Cox, Yeo-Johnson).
  - **Tame outliers** so a few extremes do not set the scale: clip or winsorize to a percentile.

  How much you owe depends on the model. Linear and distance-based models (logistic regression,
  kNN, SVM) need it; they read the geometry of the feature space directly. Tree ensembles are
  scale-invariant and mostly shrug it off, so for GBMs this is often habit more than necessity.

**What TabPFN does.** 

It rescales every numeric column itself, inside the model, before the
  forward pass: a robust scale-and-clip on some estimators, a quantile transform on others (we
  open that up at the end of the post). So the whole checklist above is preprocessing TabPFN has
  already internalized.

  The prediction: external scaling and skew transforms should not meaningfully move TabPFN's AUC
  (changes within seed and split noise), while the very same transforms move a linear model. We
  feed four input versions of `adult` (raw, standardized, quantile-normalized, log1p on the skewed
  columns) to both, and the linear model here has **no internal scaler**, so the external
  transform is the only scaling it gets.

In [9]:
from sklearn.preprocessing import QuantileTransformer

def numeric_cols(X):
  return X.select_dtypes("number").columns.tolist()

def skewed_cols(X_tr, thresh=1.0):
  """Right-skewed, nonnegative numeric columns (log1p targets); skew computed on train."""
  return [c for c in numeric_cols(X_tr)
          if (X_tr[c].dropna() >= 0).all() and X_tr[c].skew() > thresh]

def scale_standard(X_tr, X_te):
  num = numeric_cols(X_tr); sc = StandardScaler().fit(X_tr[num])
  A, B = X_tr.copy(), X_te.copy()
  A[num], B[num] = sc.transform(X_tr[num]), sc.transform(X_te[num])
  return A, B

def scale_quantile(X_tr, X_te):
  num = numeric_cols(X_tr)
  qt = QuantileTransformer(output_distribution="normal",
                           n_quantiles=min(1000, len(X_tr)), random_state=0).fit(X_tr[num])
  A, B = X_tr.copy(), X_te.copy()
  A[num], B[num] = qt.transform(X_tr[num]), qt.transform(X_te[num])
  return A, B

def transform_log1p(X_tr, X_te, thresh=1.0):
  cols = skewed_cols(X_tr, thresh)
  A, B = X_tr.copy(), X_te.copy()
  A[cols], B[cols] = np.log1p(X_tr[cols]), np.log1p(X_te[cols])
  return A, B

# Logistic regression WITHOUT an internal scaler, so the external transform is its only scaling.
def make_linear_noscale(cat_cols, num_cols):
  transformers = [("num", SimpleImputer(strategy="median"), num_cols)]
  if cat_cols:
      transformers.append(("cat", Pipeline([
          ("imp", SimpleImputer(strategy="most_frequent")),
          ("oh", OneHotEncoder(handle_unknown="infrequent_if_exist", max_categories=50))]), cat_cols))
  return Pipeline([("pre", ColumnTransformer(transformers)),
                   ("clf", LogisticRegression(max_iter=2000))])

In [10]:
import warnings

Xtr, Xte, ytr, yte, cat_cols, num_cols = prep_dataset(1590)   # adult
print("log1p targets (skew > 1, nonneg):", skewed_cols(Xtr))  

SCALERS = {"raw":      lambda a, b: (a, b),
         "standard": scale_standard,
         "quantile": scale_quantile,
         "log1p":    transform_log1p}
         
rows = []
with warnings.catch_warnings():        # the unscaled logreg does not converge; that is the point
  warnings.simplefilter("ignore")
  for vname, fn in SCALERS.items():
      Atr, Ate = fn(Xtr, Xte)
      rows.append({**evaluate(TabPFNClassifier(random_state=0), Atr, ytr, Ate, yte, "tabpfn"), "scaling": vname})
      rows.append({**evaluate(make_linear_noscale(cat_cols, num_cols), Atr, ytr, Ate, yte, "logreg_noscale"), "scaling": vname})
scaling = pd.DataFrame(rows)

log1p targets (skew > 1, nonneg): ['fnlwgt', 'capital-gain', 'capital-loss']


In [11]:
tbl = (scaling.pivot(index="scaling", columns="model", values="auc")
            .reindex(index=["raw", "standard", "quantile", "log1p"])
            .round(4))
tbl["tabpfn_d"] = (tbl["tabpfn"] - tbl.loc["raw", "tabpfn"]).round(4)          # change vs raw
tbl["logreg_d"] = (tbl["logreg_noscale"] - tbl.loc["raw", "logreg_noscale"]).round(4)
print_df(tbl, showindex=True)

+-----------+------------------+----------+------------+------------+
| scaling   |   logreg_noscale |   tabpfn |   tabpfn_d |   logreg_d |
|-----------+------------------+----------+------------+------------|
| raw       |           0.8818 |   0.9148 |     0      |     0      |
| standard  |           0.9031 |   0.9148 |     0      |     0.0213 |
| quantile  |           0.8962 |   0.9095 |    -0.0053 |     0.0144 |
| log1p     |           0.8963 |   0.9136 |    -0.0012 |     0.0145 |
+-----------+------------------+----------+------------+------------+


In [12]:
SCALERS = {"raw": lambda a, b: (a, b), "standard": scale_standard,
         "quantile": scale_quantile, "log1p": transform_log1p}

# Add the GBMs alongside tabpfn/logreg to test the invariance claim across model families.
MODELS = [("tabpfn",   lambda cc, nc: TabPFNClassifier(random_state=0)),
          ("logreg",   lambda cc, nc: make_linear_noscale(cc, nc)),
          ("lgbm",     lambda cc, nc: LGBMClassifier(random_state=0, verbose=-1)),
          ("xgb",      lambda cc, nc: make_xgb()),
          ("catboost", lambda cc, nc: make_catboost(cc))]

def scaling_sweep(data_id):
  Xtr, Xte, ytr, yte, cat_cols, num_cols = prep_dataset(data_id)
  out = {}
  with warnings.catch_warnings():
      warnings.simplefilter("ignore")
      for vname, fn in tqdm(SCALERS.items()):
          Atr, Ate = fn(Xtr, Xte)
          for mname, mk in MODELS:
              try:
                  out[(mname, vname)] = evaluate(mk(cat_cols, num_cols), Atr, ytr, Ate, yte, mname)["auc"]
              except Exception as e:
                  out[(mname, vname)] = float("nan")
                  print(f"  {data_id} {mname}/{vname} FAILED: {type(e).__name__}")
  return out

sweep = {}
for name, data_id in tqdm(PANEL.items()):
  sweep[name] = scaling_sweep(data_id)
  print(f"{name:16s} done")

 17%|█▋        | 1/6 [01:55<09:35, 115.17s/it]

credit-g         done



 33%|███▎      | 2/6 [03:19<06:27, 96.80s/it] 

adult            done



 50%|█████     | 3/6 [07:13<07:59, 159.78s/it]

...

electricity      done



 83%|████████▎ | 5/6 [09:34<01:44, 104.68s/it]

click-prediction done



100%|██████████| 6/6 [13:58<00:00, 139.77s/it]

bioresponse      done


In [13]:
# Delta AUC vs raw, per model, for each transform. The question: which model families are
# invariant. Affine (standardize) should be ~0 for every model except logreg; nonlinear
# (quantile, log1p) should be ~0 for the rank-based trees but small-nonzero for tabpfn.
MODEL_ORDER = ["logreg", "lgbm", "xgb", "catboost", "tabpfn"]

def delta_table(transform):
  recs = []
  for name, d in sweep.items():
      row = {"dataset": name}
      for m in MODEL_ORDER:
          row[m] = round(d[(m, transform)] - d[(m, "raw")], 4)
      recs.append(row)
  return pd.DataFrame(recs).set_index("dataset")

for transform, label in tqdm([("standard", "STANDARDIZE (affine)"),
                         ("quantile", "QUANTILE (nonlinear monotonic)"),
                         ("log1p",    "LOG1P (nonlinear monotonic)")]):
  print(f"\ndelta AUC vs raw, {label}:")
  print_df(delta_table(transform), showindex=True)

100%|██████████| 3/3 [00:00<00:00, 746.27it/s]


delta AUC vs raw, STANDARDIZE (affine):
+------------------+----------+---------+--------+------------+----------+
| dataset          |   logreg |    lgbm |    xgb |   catboost |   tabpfn |
|------------------+----------+---------+--------+------------+----------|
| credit-g         |  -0.0027 |  0.004  | 0      |     0      |  -0.0033 |
| adult            |   0.0212 |  0.0001 | 0      |     0      |   0      |
| bank-marketing   |   0.0022 | -0.0007 | 0      |    -0      |  -0      |
| electricity      |   0.0169 | -0.0008 | 0      |    -0      |   0.0003 |
| click-prediction |   0.0793 | -0.0016 | 0.0031 |    -0      |   0.0001 |
| bioresponse      |  -0.0349 | -0.0001 | 0      |     0.0004 |   0.0002 |
+------------------+----------+---------+--------+------------+----------+

delta AUC vs raw, QUANTILE (nonlinear monotonic):
+------------------+----------+---------+--------+------------+----------+
| dataset          |   logreg |    lgbm |    xgb |   catboost |   tabpfn |
|-------

### Footnote: you can make TabPFN fully scale-invariant

Why did the default leave a small residual under quantile and log when standardize was exactly
zero? Because half the ensemble uses the value-based `squashing_scaler` (median and IQR), which
is only affine-invariant, while the other half uses the rank-based `quantile_uni`, which is
invariant to any monotonic transform. The residual is the squashing half showing through.

Override the preprocessing so every estimator uses the rank-based transform, and that residual
should vanish: TabPFN becomes monotonic-invariant, exactly like a tree. It does, the `quant_d`
and `log_d` for `all-quantile` collapse to about zero. The honest catch is the `raw_auc` column:
forcing a single transform costs a little absolute AUC, because you give up the ensemble's
transform diversity. So this is a demonstration of the mechanism, not a knob to flip in practice.

**The measured result.** The four input versions, run across the six-dataset panel, give one
    table per transform: the delta AUC against the raw frame, so `0.0000` means the transform did
    nothing.

    Read down the `tabpfn` column. Under **standardize** it is zero to three decimals everywhere
    except credit-g, the smallest set (750 train rows), where it drifts to `-0.0033` on split noise.
    This is not approximately invariant, it is exactly invariant: an affine rescale is subtracted off
    and divided out by the internal robust scaler, so nothing new reaches the forward pass. The tree
    ensembles agree (xgboost and catboost are `0` to four decimals; lightgbm's largest move is
    `0.004`, a histogram-binning artifact rather than real sensitivity). The one model that moves is
    logistic regression: `+0.079` on click-prediction, `+0.021` on adult, `+0.017` on electricity.
    The step the checklist exists for is the only thing in the roster that needs it.

    Under **quantile** and **log1p** the `tabpfn` column is no longer exactly zero. The residual is
    small and mixed in sign (adult `-0.0054` under quantile, electricity `+0.0024`, bioresponse
    `+0.0015`; log1p is weaker because it only touches the skewed columns), but it is real. The trees
    stay flat, because a rank-preserving transform leaves their split points untouched. TabPFN does
    not, quite, and that small gap is what the footnote below is about.

    One caveat in the other direction: logistic regression's gains are conditional, not automatic.
    The same quantile transform that adds `+0.086` on click-prediction removes `-0.057` on
    bioresponse, where standardizing 1,776 low-variance molecular descriptors amplifies noise more
    than signal. "Scale your features" was always a dataset-dependent bet for linear models, and the
    panel shows both signs.

In [14]:
from tabpfn.preprocessing.configs import PreprocessorConfig

# Force every estimator onto the rank-based quantile transform (drop the value-based squashing half).
ALL_QUANTILE = {"PREPROCESS_TRANSFORMS": [PreprocessorConfig(name="quantile_uni", categorical_name="numeric")]}

Xtr, Xte, ytr, yte, cat_cols, num_cols = prep_dataset(1590)   # adult
Aq_tr, Aq_te = scale_quantile(Xtr, Xte)
Al_tr, Al_te = transform_log1p(Xtr, Xte)

rows = []
with warnings.catch_warnings():
  warnings.simplefilter("ignore")
  for label, ic in [("default", None), ("all-quantile", ALL_QUANTILE)]:
      raw = evaluate(TabPFNClassifier(random_state=0, inference_config=ic), Xtr, ytr, Xte, yte, label)["auc"]
      qd  = evaluate(TabPFNClassifier(random_state=0, inference_config=ic), Aq_tr, ytr, Aq_te, yte, label)["auc"] - raw
      ld  = evaluate(TabPFNClassifier(random_state=0, inference_config=ic), Al_tr, ytr, Al_te, yte, label)["auc"] - raw
      rows.append({"config": label, "raw_auc": round(raw, 4), "quant_d": round(qd, 4), "log_d": round(ld, 4)})
print_df(pd.DataFrame(rows).set_index("config"), showindex=True)

+--------------+-----------+-----------+---------+
| config       |   raw_auc |   quant_d |   log_d |
|--------------+-----------+-----------+---------|
| default      |    0.9148 |   -0.0054 | -0.0012 |
| all-quantile |    0.9084 |   -0.0001 | -0      |
+--------------+-----------+-----------+---------+


--- 

## Missing Values

**The familiar checklist.** A hole in the matrix is the most common defect in real tabular data, and the classical response is a ladder you climb only as far as you must:

    - **Diagnose the mechanism.** MCAR (the gap carries no information), MAR (the gap depends on
      *other observed* columns), MNAR (the gap depends on the *missing value itself*). The
      mechanism decides whether missingness is noise or signal.
    - **Drop** rows or columns when the gaps are few and uninformative.
    - **Fill** with a constant, the **median / mode**, or a group-wise statistic, learned on train.
    - **Flag** with a missing-indicator column, so the model can use "was this absent" as a
      feature. This is the standard MNAR insurance.
    - **Model** the gap with KNN or iterative (MICE) imputation when the column earns it.

    How much you owe depends on the model. Linear and distance models cannot ingest a NaN at all,
    so imputation is mandatory. Gradient boosters (LightGBM, XGBoost, native CatBoost) take NaNs
    directly and learn a default split direction, so for them imputation is often optional.

  
**What TabPFN does.**

    Leave the NaNs in. Inside the model, before the forward pass, two things happen to every
    missing cell (read straight off the v3 source):
    
    - **Mean imputation on the train context.** The cell is filled with the mean of that feature
      over the *training rows only* (`_impute_nan_and_inf_with_mean`, fit on `num_train`). Because
      the statistic is fit on the context and reused for the query rows, you cannot leak test
      information through it: the train-only discipline you would impose by hand, automatic.
    - **A missing-indicator channel.** Before imputing, TabPFN records *where* the holes were and
      feeds that pattern to the model as a parallel per-feature input (`use_nan_indicators=True`,
      a missing cell tagged `-2.0`). So informative missingness survives: the model sees both the
      filled value and the fact that it was filled.
      
    TabPFN already climbs two rungs of the ladder for you, median-style fill plus the indicator,
    with the train-only fit built in. The prediction: raw NaNs should match or beat external
    imputation; a hand-built indicator should add little, because the internal channel already
    exists; and the gap between native and impute should *widen under MNAR*, where the indicator
    is the only thing that recovers the signal.
    
    (One seam for the deep dive: that fill value is the *train* mean, reused for the query rows
    even when their distribution has shifted.)

In [ ]:
def impute_frame(Xtr, Xte, add_indicator=False):
    """
    Median-fill numerics, mode-fill categoricals; optionally append 0/1
    missing-indicator columns for every feature that had a hole in train.
    Fit on train.
    """
    num = Xtr.select_dtypes("number").columns
    cat = Xtr.select_dtypes(exclude="number").columns

    A, B = Xtr.copy(), Xte.copy()

    fills = {
        **{c: Xtr[c].median() for c in num},
        **{c: Xtr[c].mode(dropna=True)[0] for c in cat},
    }

    if add_indicator:
        for c in [c for c in Xtr.columns if Xtr[c].isna().any()]:
            A[f"{c}__isna"] = Xtr[c].isna().astype(int).values
            B[f"{c}__isna"] = Xte[c].isna().astype(int).values

    for c in num:
        A[c] = A[c].fillna(fills[c])
        B[c] = B[c].fillna(fills[c])

    for c in cat:
        # object roundtrip so the fill is robust to category-set differences
        A[c] = A[c].astype("object").fillna(fills[c]).astype("category")
        B[c] = B[c].astype("object").fillna(fills[c]).astype("category")

    return A, B


def inject_mcar(Xtr, Xte, col, frac=0.3, seed=0):
    """
    Hide a random `frac` of `col` in each split.
    Missingness carries no information.
    """
    A, B = Xtr.copy(), Xte.copy()

    for i, D in enumerate((A, B)):
        m = np.random.RandomState(seed + i).rand(len(D)) < frac
        D.loc[D.index[m], col] = np.nan

    return A, B


def inject_mnar(Xtr, Xte, col, frac=0.3, seed=0):
    """
    Self-masking MNAR: hide the largest `frac` of `col`, using a threshold
    computed from train, in both splits. Missingness now encodes
    "this value was high", which is label-informative.
    """
    thr = Xtr[col].quantile(1 - frac)

    A, B = Xtr.copy(), Xte.copy()

    A.loc[A[col] >= thr, col] = np.nan
    B.loc[B[col] >= thr, col] = np.nan

    return A, B

def inject_testonly(Xtr, Xte, col, frac=0.3, seed=0, kind="mcar"):
    """Train stays clean; inject missingness ONLY into test (the 'unseen at train' case).
    kind='mcar' hides a random frac; kind='mnar' hides the top frac by the train threshold.
    Returns (clean train, holed test)."""
    B = Xte.copy()
    if kind == "mcar":
        m = np.random.RandomState(seed).rand(len(B)) < frac
        B.loc[B.index[m], col] = np.nan
    else:
        thr = Xtr[col].quantile(1 - frac)
        B.loc[B[col] >= thr, col] = np.nan
    return Xtr.copy(), B

In [37]:
import warnings

# Post 2 experiments, all on adult. education-num is the injection target
# (strongest numeric predictor, corr +0.333). Adult's real categorical holes are
# mode-filled ONCE into a clean base (Xtr0/Xte0), so any injected hole in
# education-num is the only thing varying in B/C.


def auc_of(model, Atr, Ate):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        return float(evaluate(model, Atr, ytr, Ate, yte, "x")["auc"])

def mk(name):
    return {
        "tabpfn": TabPFNClassifier(random_state=0),
        "lgbm": LGBMClassifier(random_state=0, verbose=-1),
        "xgb": make_xgb(),
        "catboost": make_catboost(cat_cols),
        "logreg": make_linear_noscale(cat_cols, num_cols),
    }[name]


Xtr, Xte, ytr, yte, cat_cols, num_cols = prep_dataset(1590)
Xtr0, Xte0 = impute_frame(Xtr, Xte)

COL = "education-num"
rows = []

A_variants = {
    "native": (Xtr, Xte),
    "mode-impute": impute_frame(Xtr, Xte),
    "mode+indicator": impute_frame(Xtr, Xte, add_indicator=True),
}

for v, (Atr, Ate) in tqdm(A_variants.items(), desc="A real-holes"):
    for m in ["tabpfn", "lgbm", "xgb", "catboost", "logreg"]:
        rows.append(
            dict(
                exp="A",
                mechanism="real",
                model=m,
                variant=v,
                auc=float(auc_of(mk(m), Atr, Ate)),
            )
        )

for mech, inj in [("mcar", inject_mcar), ("mnar", inject_mnar)]:
    Itr, Ite = inj(Xtr0, Xte0, COL, frac=0.3, seed=0)

    B_variants = {
        "native": (Itr, Ite),
        "median": impute_frame(Itr, Ite),
        "median+indicator": impute_frame(Itr, Ite, add_indicator=True),
    }

    for v, (Atr, Ate) in tqdm(B_variants.items(), desc=f"B {mech}"):
        for m in ["tabpfn", "lgbm"]:
            rows.append(
                dict(
                    exp="B",
                    mechanism=mech,
                    model=m,
                    variant=v,
                    auc=float(auc_of(mk(m), Atr, Ate)),
                )
            )

for mech in tqdm(["mcar", "mnar"], desc="C test-only"):
    Ctr, Cte = inject_testonly(
        Xtr0,
        Xte0,
        COL,
        frac=0.3,
        seed=0,
        kind=mech,
    )

    C_variants = {
        "native": (Ctr, Cte),
        "median-impute": impute_frame(Ctr, Cte),
    }

    for v, (Atr, Ate) in C_variants.items():
        for m in ["tabpfn", "lgbm", "xgb", "catboost"]:
            rows.append(
                dict(
                    exp="C",
                    mechanism=mech,
                    model=m,
                    variant=v,
                    auc=float(auc_of(mk(m), Atr, Ate)),
                )
            )

res = pd.DataFrame(rows)
print(f"\n{len(res)} rows")
print("auc dtype:", res["auc"].dtype, "| sample:", repr(res["auc"].iloc[0]), type(res["auc"].iloc[0]))
res["auc"] = pd.to_numeric(res["auc"], errors="coerce")
print(res["auc"].describe())


C test-only: 100%|██████████| 2/2 [01:32<00:00, 46.31s/it]


43 rows
auc dtype: float64 | sample: 0.9148279419683221 <class 'numpy.float64'>
count    43.000000
mean      0.913470
std       0.011118
min       0.881832
25%       0.913111
50%       0.915415
75%       0.920481
max       0.925339
Name: auc, dtype: float64


In [38]:
print("rows:", len(res), "| coverage:")
print(res.groupby(["exp", "mechanism"]).size().to_string())


def show(exp, mech, title, col_order, row_order):
    d = res[(res.exp == exp) & (res.mechanism == mech)]

    if d.empty:
        print(f"\n{title}\n  (no rows yet — Cell 4 didn't populate this one)")
        return

    t = d.pivot_table(
        index="model",
        columns="variant",
        values="auc",
        dropna=False,
    ).round(4)

    t = t.reindex(
        index=[r for r in row_order if r in t.index],
        columns=[c for c in col_order if c in t.columns],
    )

    print(f"\n{title}")
    print_df(t, showindex=True)


show(
    "A",
    "real",
    "Exp A — adult real categorical holes (AUC):",
    ["native", "mode-impute", "mode+indicator"],
    ["tabpfn", "lgbm", "xgb", "catboost", "logreg"],
)

for mech in ["mcar", "mnar"]:
    show(
        "B",
        mech,
        f"Exp B — injected {mech.upper()} on education-num, both splits (AUC):",
        ["native", "median", "median+indicator"],
        ["tabpfn", "lgbm"],
    )

for mech in ["mcar", "mnar"]:
    show(
        "C",
        mech,
        f"Exp C — test-only {mech.upper()} on education-num (AUC):",
        ["native", "median-impute"],
        ["tabpfn", "lgbm", "xgb", "catboost"],
    )


def gap(exp, mech, base):
    d = res[(res.exp == exp) & (res.mechanism == mech)]

    if d.empty:
        return pd.Series(dtype=float)

    g = d.pivot_table(
        index="model",
        columns="variant",
        values="auc",
        dropna=False,
    )

    if "native" not in g.columns or base not in g.columns:
        return pd.Series(dtype=float)

    return (g["native"] - g[base]).round(4)


print("\nnative − plain-impute (AUC gap):")

print_df(
    pd.DataFrame(
        {
            "B/mcar": gap("B", "mcar", "median"),
            "B/mnar": gap("B", "mnar", "median"),
            "C/mcar": gap("C", "mcar", "median-impute"),
            "C/mnar": gap("C", "mnar", "median-impute"),
        }
    ),
    showindex=True,
)

rows: 43 | coverage:
exp  mechanism
A    real         15
B    mcar          6
     mnar          6
C    mcar          8
     mnar          8

Exp A — adult real categorical holes (AUC):
+----------+----------+---------------+------------------+
| model    |   native |   mode-impute |   mode+indicator |
|----------+----------+---------------+------------------|
| tabpfn   |   0.9148 |        0.9138 |           0.9148 |
| lgbm     |   0.9228 |        0.9213 |           0.9219 |
| xgb      |   0.9161 |        0.9154 |           0.917  |
| catboost |   0.925  |        0.9243 |           0.9253 |
| logreg   |   0.8818 |        0.8818 |           0.8818 |
+----------+----------+---------------+------------------+

Exp B — injected MCAR on education-num, both splits (AUC):
+---------+----------+----------+--------------------+
| model   |   native |   median |   median+indicator |
|---------+----------+----------+--------------------|
| tabpfn  |   0.9134 |   0.9134 |             0.9135 |
| l

In [39]:
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import r2_score
from lightgbm import LGBMRegressor

X, y = load_openml(1590)
num = X.select_dtypes("number").columns.tolist()
cat = X.select_dtypes(exclude="number").columns.tolist()
Xe = X.copy()
Xe[cat] = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1,
                       encoded_missing_value=-2).fit_transform(X[cat].astype("object"))

recs = []
for tgt in tqdm(num, desc="reconstructability"):
    feats = [c for c in Xe.columns if c != tgt]
    Atr, Ate, btr, bte = train_test_split(Xe[feats], Xe[tgt], test_size=0.3, random_state=0)
    r2 = r2_score(bte, LGBMRegressor(n_estimators=300, learning_rate=0.05, verbose=-1)
                       .fit(Atr, btr).predict(Ate))
    recs.append({"column": tgt,
                 "corr_with_label": round(abs(np.corrcoef(X[tgt], y)[0, 1]), 3),
                 "R2_from_rest": round(r2, 3)})
    
redund = pd.DataFrame(recs).sort_values("R2_from_rest").reset_index(drop=True)
print_df(redund)

reconstructability: 100%|██████████| 6/6 [00:02<00:00,  2.14it/s]

+----------------+-------------------+----------------+
| column         |   corr_with_label |   R2_from_rest |
|----------------+-------------------+----------------|
| capital-loss   |             0.148 |          0.01  |
| fnlwgt         |             0.006 |          0.053 |
| capital-gain   |             0.223 |          0.068 |
| hours-per-week |             0.228 |          0.267 |
| age            |             0.23  |          0.489 |
| education-num  |             0.333 |          1     |
+----------------+-------------------+----------------+


In [32]:
print("rows:", len(res))
print(res.groupby(["exp", "mechanism"]).size().to_string())

rows: 43
exp  mechanism
A    real         15
B    mcar          6
     mnar          6
C    mcar          8
     mnar          8
